# Control held-out pareado: predictor identidad vs. Xavier

Este notebook reconstruye los checkpoints congelados de ambas condiciones y, sólo si los diez replays pasan, compara predicción, clustering y no-colapso sobre las mismas 144 secuencias de test.

**Estado al preparar este archivo:** sin ejecutar y sin outputs. La condición identidad ya consumió esta partición en un protocolo anterior; por eso el resultado será una comparación predeclarada, no una segunda prueba ciega independiente.

## Hipótesis y criterios congelados

La hipótesis es que cambiar únicamente `M₀ = I` por Xavier uniforme puede producir un operador final denso y no-identidad sin destruir la calidad predictiva ni el clustering de los regímenes. El protocolo está congelado en `configs/paper_linear_random_heldout_smoke.yaml`.

Para cada seed, random debe satisfacer simultáneamente:

- error predictivo normalizado no mayor que `2×` el de identidad;
- pureza K-means al menos `50%` y al menos `90%` de la pureza identidad;
- rango efectivo al menos `4` y al menos `50%` del rango identidad.

La pureza usa labels sólo para puntuar clusters ajustados sin supervisión. La matched accuracy se reporta como diagnóstico. Ningún threshold puede modificarse después de ejecutar la celda que construye test.

In [ ]:
# ruff: noqa: E402, E501
import json
import platform
import random
import sys
import time
from dataclasses import asdict, replace
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "No se encontró la raíz del repositorio."
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

from koopman_jepa.paper_config import load_paper_linear_random_heldout_config
from koopman_jepa.paper_data import (
    PAPER_REGIME_NAMES,
    PaperRegimeDataset,
    generate_paper_master,
)
from koopman_jepa.paper_evaluation import (
    evaluate_paper_linear_paired_heldout,
    evaluate_paper_linear_random_heldout_gate,
    evaluate_paper_linear_structure,
)
from koopman_jepa.paper_model import PaperTemporalJEPA
from koopman_jepa.paper_training import (
    make_paper_loader,
    run_paper_train_validation_with_checkpoint,
    verify_paper_checkpoint_replay,
)

plt.style.use("seaborn-v0_8-whitegrid")
print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
config_path = ROOT / "configs" / "paper_linear_random_heldout_smoke.yaml"
config = load_paper_linear_random_heldout_config(config_path)
identity_model_config = replace(config.model, linear_initialization="identity")
torch.use_deterministic_algorithms(True)

print(config_path.relative_to(ROOT))
print(json.dumps(asdict(config), indent=2))

## Reconstrucción sin acceso a test

Las próximas dos celdas sólo construyen train/validation y reconstruyen los diez checkpoints. También verifican que los encoders iniciales sean exactamente iguales dentro de cada par y que las matrices predictor iniciales sean diferentes.

In [ ]:
train_dataset = PaperRegimeDataset(config.data, "train", generate_paper_master)
validation_dataset = PaperRegimeDataset(config.data, "val", generate_paper_master)
train_keys = {train_dataset.sample_key(index) for index in range(len(train_dataset))}
validation_keys = {
    validation_dataset.sample_key(index) for index in range(len(validation_dataset))
}

assert len(train_dataset) == 32 * len(PAPER_REGIME_NAMES) == 576
assert len(validation_dataset) == 8 * len(PAPER_REGIME_NAMES) == 144
assert train_keys.isdisjoint(validation_keys)
print(f"Train/validation: {len(train_dataset)}/{len(validation_dataset)}")
print("Intersección train/validation: 0 — PASS")
print("Test todavía no fue instanciado.")

In [ ]:
def clone_initial_state(model):
    return {
        "online": {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.online_encoder.state_dict().items()
        },
        "target": {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.target_encoder.state_dict().items()
        },
        "predictor": model.predictor.matrix.detach().cpu().clone(),
    }


def encoders_match(observed, reference):
    return all(
        torch.equal(tensor, reference[module_name][name])
        for module_name in ("online", "target")
        for name, tensor in observed[module_name].items()
    )


def replay_condition(name, model_config, replay_config, reference_states=None):
    expected_epochs = dict(
        zip(config.sweep.seeds, replay_config.expected_epochs, strict=True)
    )
    models = {}
    results = {}
    initial_states = {}
    pairing_checks = {}
    times = {}

    for seed in config.sweep.seeds:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        run_config = replace(config.train, seed=seed)
        model = PaperTemporalJEPA(model_config)
        initial_state = clone_initial_state(model)
        initial_states[seed] = initial_state
        if reference_states is None:
            pairing_checks[seed] = True
        else:
            pairing_checks[seed] = (
                encoders_match(initial_state, reference_states[seed])
                and not torch.equal(
                    initial_state["predictor"],
                    reference_states[seed]["predictor"],
                )
            )

        started = time.perf_counter()
        checkpoint_run = run_paper_train_validation_with_checkpoint(
            model,
            train_dataset,
            validation_dataset,
            run_config,
            config.checkpoint_gate,
        )
        replay = verify_paper_checkpoint_replay(
            model,
            checkpoint_run,
            validation_dataset,
            run_config,
            replay_config,
            expected_epoch=expected_epochs[seed],
        )
        times[seed] = time.perf_counter() - started
        models[seed] = model
        results[seed] = replay
        selected_epoch = (
            checkpoint_run.selection.epoch
            if checkpoint_run.selection is not None
            else None
        )
        print(
            f"{name} seed={seed} expected={expected_epochs[seed]} "
            f"selected={selected_epoch} pairing={pairing_checks[seed]} "
            f"replay={'PASS' if replay.passed else 'FAIL'} "
            f"time={times[seed]:.2f}s"
        )

    return models, results, initial_states, pairing_checks, times


identity_models, identity_replays, identity_initial_states, _, identity_times = (
    replay_condition(
        "identity",
        identity_model_config,
        config.identity_replay,
    )
)
random_models, random_replays, _, paired_initializations, random_times = (
    replay_condition(
        "random",
        config.model,
        config.random_replay,
        reference_states=identity_initial_states,
    )
)
identity_replay_passed = (
    set(identity_replays) == set(config.sweep.seeds)
    and all(result.passed for result in identity_replays.values())
)
random_replay_passed = (
    set(random_replays) == set(config.sweep.seeds)
    and all(result.passed for result in random_replays.values())
)
pairing_passed = (
    set(paired_initializations) == set(config.sweep.seeds)
    and all(paired_initializations.values())
)
replay_gate_open = identity_replay_passed and random_replay_passed and pairing_passed
assert replay_gate_open, (
    "ABORTAR: algún checkpoint o pareo inicial no coincide. "
    "Test no debe construirse."
)
print(f"Identity replay: PASS; total={sum(identity_times.values()):.2f}s")
print(f"Random replay: PASS; total={sum(random_times.values()):.2f}s")
print("La próxima celda es la única que puede construir test.")

## Apertura del test ya consumido

La celda siguiente falla antes de construir test si cualquiera de los diez replays o el pareo de inicialización no pasó. Desde su primera ejecución, esta evaluación debe tratarse como consumida para el control random.

In [ ]:
assert globals().get("replay_gate_open", False), (
    "El replay gate no está abierto; no se permite construir test."
)
test_dataset = PaperRegimeDataset(config.data, "test", generate_paper_master)
test_keys = {test_dataset.sample_key(index) for index in range(len(test_dataset))}

assert len(test_dataset) == 8 * len(PAPER_REGIME_NAMES) == 144
assert train_keys.isdisjoint(test_keys)
assert validation_keys.isdisjoint(test_keys)
print(f"Test construido: {len(test_dataset)} secuencias")
print("Intersecciones con train y validation: 0 — PASS")

In [ ]:
@torch.no_grad()
def collect_paired_embeddings(model, dataset, run_config):
    device = torch.device(run_config.device)
    model.to(device)
    model.eval()
    loader = make_paper_loader(dataset, run_config, shuffle=False)
    online_chunks = []
    target_chunks = []
    label_chunks = []
    for context, target, labels in loader:
        online_chunks.append(model.online_encoder(context.to(device)).cpu().numpy())
        target_chunks.append(model.target_encoder(target.to(device)).cpu().numpy())
        label_chunks.append(labels.numpy())
    return (
        np.concatenate(online_chunks, axis=0),
        np.concatenate(target_chunks, axis=0),
        np.concatenate(label_chunks, axis=0),
    )


identity_metrics_by_seed = {}
random_metrics_by_seed = {}
random_structures_by_seed = {}
for seed in config.sweep.seeds:
    run_config = replace(config.train, seed=seed)
    identity_online, identity_target, identity_labels = collect_paired_embeddings(
        identity_models[seed], test_dataset, run_config
    )
    random_online, random_target, random_labels = collect_paired_embeddings(
        random_models[seed], test_dataset, run_config
    )
    assert np.array_equal(identity_labels, random_labels)

    identity_matrix = identity_models[seed].predictor.matrix.detach().cpu().numpy()
    random_matrix = random_models[seed].predictor.matrix.detach().cpu().numpy()
    identity_metrics_by_seed[seed] = evaluate_paper_linear_paired_heldout(
        identity_matrix,
        identity_online,
        identity_target,
        identity_labels,
        config.evaluation,
    )
    random_metrics_by_seed[seed] = evaluate_paper_linear_paired_heldout(
        random_matrix,
        random_online,
        random_target,
        random_labels,
        config.evaluation,
    )
    random_structures_by_seed[seed] = evaluate_paper_linear_structure(
        random_matrix
    )

gate = evaluate_paper_linear_random_heldout_gate(
    identity_metrics_by_seed,
    random_metrics_by_seed,
    config.sweep,
    config.gate,
)
print(json.dumps(asdict(gate), indent=2))

In [ ]:
comparison_by_seed = {row.seed: row for row in gate.comparisons}
print("seed pred_I pred_R ratio purity_I purity_R ratio rank_I rank_R ratio matched_R id_error offdiag")
for seed in config.sweep.seeds:
    row = comparison_by_seed[seed]
    structure = random_structures_by_seed[seed]
    print(
        f"{seed:>4d} {row.identity_prediction_error:>6.3f} "
        f"{row.random_prediction_error:>6.3f} {row.prediction_error_ratio:>5.2f} "
        f"{row.identity_kmeans_purity:>8.2%} {row.random_kmeans_purity:>8.2%} "
        f"{row.kmeans_purity_ratio:>5.2f} {row.identity_effective_rank:>6.2f} "
        f"{row.random_effective_rank:>6.2f} {row.effective_rank_ratio:>5.2f} "
        f"{row.random_kmeans_matched_accuracy:>9.2%} "
        f"{structure.relative_identity_error:>8.2%} "
        f"{structure.off_diagonal_fraction:>7.2%}"
    )

## Comparación seed por seed

Las barras pareadas muestran los valores absolutos de cada sistema latente; los paneles de ratios aplican los criterios invariantes de escala o relativos a identidad. Las líneas rojas son los límites congelados.

In [ ]:
seeds = np.array(config.sweep.seeds)
rows = [comparison_by_seed[seed] for seed in seeds]
identity_prediction = np.array([row.identity_prediction_error for row in rows])
random_prediction = np.array([row.random_prediction_error for row in rows])
prediction_ratio = np.array([row.prediction_error_ratio for row in rows])
identity_purity = np.array([row.identity_kmeans_purity for row in rows])
random_purity = np.array([row.random_kmeans_purity for row in rows])
purity_ratio = np.array([row.kmeans_purity_ratio for row in rows])
identity_rank = np.array([row.identity_effective_rank for row in rows])
random_rank = np.array([row.random_effective_rank for row in rows])
rank_ratio = np.array([row.effective_rank_ratio for row in rows])

fig, axes = plt.subplots(2, 3, figsize=(17, 10), constrained_layout=True)
width = 0.36

axes[0, 0].bar(seeds - width / 2, identity_prediction, width=width, label="identidad")
axes[0, 0].bar(seeds + width / 2, random_prediction, width=width, label="random")
axes[0, 0].set(title="Error predictivo normalizado", xlabel="Seed", ylabel="Error")
axes[0, 0].legend()

axes[0, 1].bar(seeds, prediction_ratio, color="tab:purple")
axes[0, 1].axhline(config.gate.max_random_to_identity_prediction_error_ratio, color="tab:red", linestyle="--")
axes[0, 1].set(title="Ratio de error random/identidad", xlabel="Seed", ylabel="Ratio")

axes[0, 2].bar(seeds - width / 2, identity_purity, width=width, label="identidad")
axes[0, 2].bar(seeds + width / 2, random_purity, width=width, label="random")
axes[0, 2].axhline(config.gate.min_random_kmeans_purity, color="tab:red", linestyle="--")
axes[0, 2].set(title="Pureza K-means", xlabel="Seed", ylabel="Pureza")
axes[0, 2].legend()

axes[1, 0].bar(seeds, purity_ratio, color="tab:green")
axes[1, 0].axhline(config.gate.min_random_to_identity_kmeans_purity_ratio, color="tab:red", linestyle="--")
axes[1, 0].set(title="Retención de pureza random/identidad", xlabel="Seed", ylabel="Ratio")

axes[1, 1].bar(seeds - width / 2, identity_rank, width=width, label="identidad")
axes[1, 1].bar(seeds + width / 2, random_rank, width=width, label="random")
axes[1, 1].axhline(config.gate.min_random_test_effective_rank, color="tab:red", linestyle="--")
axes[1, 1].set(title="Rango efectivo en test", xlabel="Seed", ylabel="Rango")
axes[1, 1].legend()

axes[1, 2].bar(seeds, rank_ratio, color="tab:orange")
axes[1, 2].axhline(config.gate.min_random_to_identity_effective_rank_ratio, color="tab:red", linestyle="--")
axes[1, 2].set(title="Retención de rango random/identidad", xlabel="Seed", ylabel="Ratio")

plt.show()

In [ ]:
status = "PASS" if gate.passed else "FAIL"
criteria = {
    "cinco pares presentes": gate.all_seeds_present,
    "métricas finitas": gate.all_finite,
    "predicción relativa": gate.prediction_passed,
    "pureza absoluta": gate.absolute_purity_passed,
    "retención de pureza": gate.relative_purity_passed,
    "rango absoluto": gate.absolute_rank_passed,
    "retención de rango": gate.relative_rank_passed,
}
failed_criteria = [name for name, passed in criteria.items() if not passed]
failed_text = ", ".join(failed_criteria) if failed_criteria else "ninguno"
failed_seed_text = ", ".join(str(seed) for seed in gate.failed_seeds) or "ninguna"

pressures = {
    "predicción relativa": gate.worst_prediction_error_ratio / config.gate.max_random_to_identity_prediction_error_ratio,
    "pureza absoluta": config.gate.min_random_kmeans_purity / max(gate.minimum_random_kmeans_purity, 1e-12),
    "retención de pureza": config.gate.min_random_to_identity_kmeans_purity_ratio / max(gate.minimum_kmeans_purity_ratio, 1e-12),
    "rango absoluto": config.gate.min_random_test_effective_rank / max(gate.minimum_random_test_effective_rank, 1e-12),
    "retención de rango": config.gate.min_random_to_identity_effective_rank_ratio / max(gate.minimum_effective_rank_ratio, 1e-12),
}
closest_criterion = max(pressures, key=pressures.get)
identity_purity_mean = identity_purity.mean()
random_purity_mean = random_purity.mean()
identity_rank_mean = identity_rank.mean()
random_rank_mean = random_rank.mean()

decision = (
    "El control held-out pareado pasa. Esto completa evidencia smoke de que una base latente alternativa puede conservar predicción y clustering con un operador denso no-identidad. El siguiente paso es decidir si escalamos la condición o si resolvemos primero las sensibilidades del dataset."
    if gate.passed
    else "El control held-out pareado falla. Debemos reportar qué seed y criterio fallaron sin retocar epochs, umbrales ni esta partición consumida."
)

display(Markdown(f"""## Resultado e interpretación

- **Gate global: {status}.**
- **Criterios fallidos:** {failed_text}.
- **Seeds fallidas:** {failed_seed_text}.
- **Peor ratio de error predictivo:** `{gate.worst_prediction_error_ratio:.3f}` frente al máximo `{config.gate.max_random_to_identity_prediction_error_ratio:.1f}`.
- **Mínima pureza random:** `{gate.minimum_random_kmeans_purity:.2%}` frente al mínimo `{config.gate.min_random_kmeans_purity:.0%}`.
- **Mínima retención de pureza:** `{gate.minimum_kmeans_purity_ratio:.3f}` frente al mínimo `{config.gate.min_random_to_identity_kmeans_purity_ratio:.2f}`.
- **Mínimo rango random:** `{gate.minimum_random_test_effective_rank:.2f}` frente al mínimo `{config.gate.min_random_test_effective_rank:.1f}`.
- **Mínima retención de rango:** `{gate.minimum_effective_rank_ratio:.3f}` frente al mínimo `{config.gate.min_random_to_identity_effective_rank_ratio:.2f}`.

### Lectura de los gráficos

Los paneles absolutos evitan que dos condiciones igualmente malas pasen por parecerse; los paneles de ratios preguntan si random conserva el comportamiento de su par identidad. El criterio relativamente más cercano a su frontera es **{closest_criterion}**. La pureza media cambia de `{identity_purity_mean:.2%}` en identidad a `{random_purity_mean:.2%}` en random; el rango efectivo medio cambia de `{identity_rank_mean:.2f}` a `{random_rank_mean:.2f}`.

El error predictivo está normalizado por la energía del target dentro de cada sistema latente, por lo que no premia ni castiga una simple diferencia global de escala. Pureza y rango responden preguntas distintas: la primera mide separación por régimen y el segundo descarta que esa separación aparente provenga de una representación casi colapsada.

### Decisión

{decision}

La interpretación sigue limitada por 8 secuencias de test por régimen, la equivalencia observacional entre `Sine_MedFreq` y `Sine_LowAmp`, la falta de hiperparámetros publicados y el uso previo de la condición identidad sobre este mismo test. No es una reproducción paper-scale.
"""))